# HW2 - Width and Weight Changes

This experiment measures how weight changes depend on network width. Train
networks with two ReLU hidden layers on the same ten data pairs.
Then inspect the middle weight matrix and measure its change across widths.

Complete the functions marked `TODO` and run the cells in order. The experiment
uses five initializations at each width to show how the results vary between runs.

Reference: [The Neural Tangent Kernel — RBC Borealis](https://rbcborealis.com/research-blogs/the-neural-tangent-kernel/).


## Part 0 - Setup

Generate ten input–target pairs by drawing each input and target independently from
$\mathcal{N}\left(0,1\right)$. Every run uses these same pairs, so the width comparisons
share one training task.

In [ ]:
import copy
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)

WIDTHS = (10, 25, 50, 100, 250, 500, 750, 1000)
DISPLAY_WIDTHS = (10, 50, 1000)
SEEDS = tuple(range(5))
STEPS = 10000
LR = 0.05
MOMENTUM = 0.9
BATCH_SIZE = 5

data_generator = torch.Generator().manual_seed(1234)
x_train = torch.randn(10, generator=data_generator)
y_train = torch.randn(10, generator=data_generator)

## Part 1 - Define the network

Both hidden layers have width $m$. Use the following network:
$$
\begin{aligned}
\boldsymbol{h}_1
&=\operatorname{ReLU}\left(\boldsymbol{w}_0 x+\boldsymbol{b}_0\right),\\
\boldsymbol{h}_2
&=\frac{1}{\sqrt{m}}\operatorname{ReLU}
\left(\boldsymbol{W}_1\boldsymbol{h}_1+\boldsymbol{b}_1\right),\\
f_{\boldsymbol{\theta}}\left(x\right)
&=\frac{1}{\sqrt{m}}
\left(\boldsymbol{a}^{\top}\boldsymbol{h}_2+b_2\right).
\end{aligned}
$$
The factors $1/\sqrt{m}$ keep the initial output scale comparable across widths.
The vector $\boldsymbol{\theta}$ contains all weights and biases, which are trained.

In `__init__`, create the layers `input = nn.Linear(1, width)`,
`hidden = nn.Linear(width, width)`, and `output = nn.Linear(width, 1)`.
In `forward`, implement the equations above. Inputs and outputs have shape `(N,)`.

The supplied `initialize` function draws the weights independently from
$\mathcal{N}\left(0,1\right)$ and sets the biases to zero.
Run the supplied checks to verify the output shape and count the parameters at each width.

In [ ]:
class WideNetwork(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.width = width
        # TODO (15 points): define self.input, self.hidden, and self.output.
        raise NotImplementedError

    def forward(self, x):
        # TODO (20 points): implement both ReLU layers and both width rescalings.
        raise NotImplementedError

In [ ]:
def initialize(width, seed):
    model = WideNetwork(width)
    generator = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        for layer in (model.input, model.hidden, model.output):
            values = torch.randn(layer.weight.shape, generator=generator)
            layer.weight.copy_(values)
            layer.bias.zero_()
    return model


print("width   number of parameters")
for width in WIDTHS:
    check_model = initialize(width, seed=0)
    count = sum(parameter.numel() for parameter in check_model.parameters())
    assert count == width**2 + 4 * width + 1
    assert check_model(x_train).shape == y_train.shape
    print(f"{width:5d}   {count:20,d}")

## Part 2 - Train the networks

Implement one training update in `train_step`. Clear the old gradients, compute the
batch loss, backpropagate, and update the parameters. Use half the mean squared error:
$$
\ell_{\mathrm{batch}}
=\frac{1}{2B}\sum_{i=1}^{B}
\left(f_{\boldsymbol{\theta}}\left(x_i\right)-y_i\right)^2.
$$

The driver applies 10,000 SGD updates with learning rate $0.05$, momentum $0.9$,
and five training pairs per batch. These settings are the same at every width.
Runs with the same seed also use the same sequence of batches.

The driver saves the network before training as `initial` and after training as
`trained`. The table reports the mean loss before and after training. The saved
networks will let us measure how far the weights moved.

In [ ]:
def train_step(model, optimizer, x, y):
    """Take one update on half the mean squared error of this batch."""
    # TODO (30 points): complete one training update.
    raise NotImplementedError

In [ ]:
def run(width, seed):
    trained = initialize(width, seed)
    initial = copy.deepcopy(trained)
    optimizer = torch.optim.SGD(trained.parameters(), lr=LR, momentum=MOMENTUM)
    batch_generator = torch.Generator().manual_seed(100 + seed)

    with torch.no_grad():
        initial_loss = 0.5 * (initial(x_train) - y_train).square().mean().item()
    for step in range(STEPS):
        indices = torch.randperm(len(x_train), generator=batch_generator)[:BATCH_SIZE]
        train_step(trained, optimizer, x_train[indices], y_train[indices])
    with torch.no_grad():
        final_loss = 0.5 * (trained(x_train) - y_train).square().mean().item()

    return {"width": width, "seed": seed, "initial": initial, "trained": trained,
            "initial_loss": initial_loss, "final_loss": final_loss}

In [ ]:
runs = {}
start_time = time.perf_counter()
print(f"Mean loss over {len(SEEDS)} initializations")
print("width    before training    after training")
for width in WIDTHS:
    for seed in SEEDS:
        runs[width, seed] = run(width, seed)
    initial_loss = np.mean([runs[width, seed]["initial_loss"] for seed in SEEDS])
    final_loss = np.mean([runs[width, seed]["final_loss"] for seed in SEEDS])
    print(f"{width:5d}    {initial_loss:15.4f}    {final_loss:14.4f}")
print(f"Total training time: {time.perf_counter() - start_time:.1f} seconds")

## Part 3 - Examine the weight changes

Now compare the weights before and after training.

Focus on `model.hidden.weight`, the matrix connecting the two hidden layers.
Write $\boldsymbol{W}_1^0$ for its initial value and $\boldsymbol{W}_1^T$ for its
value after training. Implement `weight_statistics` to return, in order,
$$
\underbrace{\left\|\boldsymbol{W}_1^0\right\|_{\mathrm F}}_{\text{initial norm}},
\qquad
\underbrace{\left\|\boldsymbol{W}_1^T-\boldsymbol{W}_1^0\right\|_{\mathrm F}}_{\text{change norm}},
\qquad
\underbrace{\frac{\left\|\boldsymbol{W}_1^T-\boldsymbol{W}_1^0\right\|_{\mathrm F}}
{\left\|\boldsymbol{W}_1^0\right\|_{\mathrm F}}}_{\text{relative change}}.
$$
Return three floats. The Frobenius norm is the square root of the sum of squared
matrix entries.

The first figure displays the full matrices at widths $10,50,1000$ for seed $0$.
The initial and trained matrices share one color scale. The changes share another
scale across all three widths, making their sizes directly comparable.

In [ ]:
def weight_statistics(initial_weights, trained_weights):
    """Return (initial norm, change norm, relative change) as floats."""
    # TODO (35 points): Compute the three weight statistics.
    raise NotImplementedError

In [ ]:
for result in runs.values():
    W0 = result["initial"].hidden.weight.detach()
    WT = result["trained"].hidden.weight.detach()
    initial_norm, change_norm, relative_change = weight_statistics(W0, WT)
    result["initial_norm"] = initial_norm
    result["change_norm"] = change_norm
    result["relative_change"] = relative_change

matrices = []
for width in DISPLAY_WIDTHS:
    W0 = runs[width, 0]["initial"].hidden.weight.detach()
    WT = runs[width, 0]["trained"].hidden.weight.detach()
    matrices.append((W0, WT, WT - W0))
weight_limit = max(matrix.abs().max().item() for row in matrices for matrix in row[:2])
change_limit = max(row[2].abs().max().item() for row in matrices)

fig, axes = plt.subplots(3, 3, figsize=(10, 8), constrained_layout=True)
for row, (width, values) in enumerate(zip(DISPLAY_WIDTHS, matrices)):
    for col, matrix in enumerate(values):
        limit = weight_limit if col < 2 else change_limit
        im = axes[row, col].imshow(matrix.numpy(), cmap="coolwarm",
                                    vmin=-limit, vmax=limit, interpolation="nearest")
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(f"Width {width}")
        if col == 1:
            weight_image = im
        if col == 2:
            change_image = im
for ax, title in zip(axes[0], ("Before training", "After training", "Change")):
    ax.set_title(title)
fig.colorbar(weight_image, ax=axes[:, :2].ravel().tolist(), shrink=0.75,
             label="Weight value")
fig.colorbar(change_image, ax=axes[:, 2].ravel().tolist(), shrink=0.75,
             label="Weight change")
plt.show()

### Compare the size of the changes

The images show individual entries. The next plots summarize the whole matrix:
its initial norm, the norm of its change, and their ratio. All three plots use
linear axes. Gray dots show the five initializations, and each colored line shows
their mean.

Read the first two plots together. Each matrix has $m^2$ entries initialized with
variance $1$, so its initial norm grows approximately as $m$. Compare this growth
with the size of the training update. The third plot shows how large the update is
relative to the starting matrix.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
keys = ("initial_norm", "change_norm", "relative_change")
labels = ("Initial weight norm", "Norm of weight change", "Relative weight change")
colors = ("tab:red", "tab:cyan", "tab:purple")
for ax, key, label, color in zip(axes, keys, labels, colors):
    means = []
    for index, width in enumerate(WIDTHS):
        values = [runs[width, seed][key] for seed in SEEDS]
        ax.scatter([width] * len(values), values, color="0.7", s=18,
                   label="individual initializations" if index == 0 else None)
        means.append(np.mean(values))
    ax.plot(WIDTHS, means, "o-", color=color, markersize=4, label="mean")
    ax.set(xlabel="Width", ylabel=label, xlim=(0, 1050), ylim=(0, None))
    ax.set_xticks([0, 250, 500, 750, 1000])
    ax.grid(alpha=0.2)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"Middle weight matrix: means over {len(SEEDS)} initializations")
print("width    initial norm    change norm    relative change")
for width in WIDTHS:
    values = [np.mean([runs[width, seed][key] for seed in SEEDS]) for key in keys]
    print(f"{width:5d}    {values[0]:12.3f}    {values[1]:11.3f}    {values[2]:15.4f}")

## What to notice

The initial norm grows approximately in proportion to width, while the norm of the change remains on a similar scale. Their ratio therefore falls sharply.